In [ ]:
import torch
import torch.nn as nn
from einops import rearrange
# Adaptive Channel-aware attention block
class CAM(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.prob = nn.Softmax(dim=-1)
        self.query_conv = nn.Conv2d(in_channels=dim, out_channels=dim, kernel_size=1)
        self.key_conv = nn.Conv2d(in_channels=dim, out_channels=dim, kernel_size=1)
        self.gamma = nn.Parameter(torch.zeros(1))
        self.value_conv = nn.Conv2d(in_channels=dim, out_channels=dim, kernel_size=1)

    def forward(self, x):
        B, C, H, W = x.shape

        # Project Q, K, V
        proj_query = self.query_conv(x)  # (B, C, H, W)
        proj_key = self.key_conv(x)      # (B, C, H, W)
        proj_value = self.value_conv(x)  # (B, C, H, W)

        # Reshape
        proj_query = rearrange(proj_query, 'b c h w -> b c (h w)')  # (B, C, H*W)
        proj_key = rearrange(proj_key, 'b c h w -> b c (h w)')      # (B, C, H*W)
        proj_value = rearrange(proj_value, 'b c h w -> b c (h w)')  # (B, C, H*W)

        # cosine similarity
        dots = torch.bmm(proj_query, proj_key.transpose(1, 2))  # (B, C, C)
        query_norm = torch.norm(proj_query, p=2, dim=-1, keepdim=True)  # (B, C, 1)
        key_norm = torch.norm(proj_key, p=2, dim=-1, keepdim=True)       # (B, C, 1)
        scale = torch.bmm(query_norm, key_norm.transpose(1, 2))  # (B, C, C)
        cosine_sim = dots / scale

        attention = self.prob(cosine_sim)  # (B, C, C)

        out = torch.bmm(attention, proj_value)  # (B, C, H*W)
        out = rearrange(out, 'b c (h w) -> b c h w', h=H, w=W)  # (B, C, H, W)
        out = self.gamma * out + x  # Skip connection

        return out
# Adaptive Spatial-aware attention block
class PAM(nn.Module):
    def __init__(self, dim ):
        super().__init__()
        self.pw = nn.Conv2d(dim, dim, kernel_size=1)
        self.prob = nn.Softmax(dim=1)
        self.query_conv = nn.Conv2d(in_channels=dim, out_channels=dim//8, kernel_size=1)
        self.key_conv = nn.Conv2d(in_channels=dim, out_channels=dim//8, kernel_size=1)
        self.value_conv = nn.Conv2d(in_channels=dim, out_channels=dim, kernel_size=1)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, C, H, W = x.shape
        proj_query = self.query_conv(x)
        proj_key = self.key_conv(x)
        proj_value = self.value_conv(x)
        proj_query = rearrange(proj_query, 'b c h w -> b (h w) c')  # (B, H*W, C)
        proj_key = rearrange(proj_key, 'b c h w -> b c (h w)')    # (B, C, H*W)

        energy = torch.bmm(proj_query, proj_key)  # (B, H*W, H*W)
        attention = self.prob(energy)
        proj_value = rearrange(proj_value, 'b c h w -> b c (h w)')
        out = torch.bmm(proj_value, rearrange(attention, 'b h w -> b w h'))
        out = rearrange(out, 'b c (h w) -> b c h w', h=H, w=W)  # (B, C, H, W)
        out = self.gamma*out + x

        return out
# Cross-Dimensional Attention (CDA)
class HybridAtten(nn.Module):
    def __init__(self, dim=64):
        super(HybridAtten, self).__init__()

        self.sa = PAM(dim)
        self.sc = CAM(dim)
    def forward(self, x):
        # First branch: positional attention
        sa_feat = self.sa(x)
        # Second branch: channel attention
        sc_feat = self.sc(sa_feat)
        # Combine features
        feat_sum = sa_feat + sc_feat

        return feat_sum